In [150]:
import pandas as pd
import numpy as np
from pathlib import Path

In [151]:
DATA = Path("../data")

In [152]:
tickets = pd.read_csv(DATA / "tickets.csv")

In [153]:
tickets.head()

,ticket_id,created_at,first_response_at,resolved_at,status,channel,customer_id,order_id,product_sku,category,...,assigned_team,agent_id,transfers,csat_score,refund_amount_inr,refund_reason_code,replacement_issued,customer_message,agent_notes,source_system
0,TK-240001,2025-01-01 06:12,2025-01-01 06:20,2025-01-02 06:33,resolved,voice,C102873,NaN,VA-EB-AIR,Billing & Payments,...,Billing,A3033,0,3.0,764.0,RETURN-QC-OK,N,[IVR transcript] you picked up the item 4 days...,re: refund pending | chk reverse pickup qc sta...,legacy_fd
1,TK-240002,2025-01-01 09:01,2025-01-01 09:34,2025-01-01 09:45,resolved,social,C103321,NaN,VA-HP-ST3,Other,...,Chat Frontline,A3015,0,0.0,NaN,NaN,N,product: my strata 3 headphones\norder: vr8959...,Reported: promo code invalid. Checked coupon v...,legacy_fd
2,TK-240004,2025-01-02 00:41,2025-01-02 03:41,2025-01-04 03:59,resolved,email,C100687,VR895382,VA-EB-PL1,Delivery & Shipping,...,Logistics,A3028,0,4.0,NaN,NaN,N,"hi there,\norder vr895382 (my pulse). i haven'...",Cust contacted re order not delivered. Confiir...,legacy_fd
3,TK-240006,2025-01-02 02:21,2025-01-02 03:38,2025-01-12 04:37,resolved,social,C104148,NaN,VA-EB-PL1,Audio Quality,...,Chat Frontline,A3044,1,3.0,NaN,NaN,N,honestly regretting this purchase. bought the ...,1. Audio distortion\n2. Asked for a recording\...,legacy_fd
4,TK-240007,2025-01-02 04:05,2025-01-02 04:38,2025-01-02 05:09,resolved,social,C103733,VR883581,VA-EB-PL1,Billing & Payments,...,Billing,A3035,0,0.0,NaN,NaN,N,"paid via UPI, amount deducted, no order confir...",cx: amount deducted without order | asked for ...,legacy_fd


In [154]:
tickets.shape

(11816, 21)

In [155]:
tickets["ticket_id"].nunique()

11200

In [156]:
tickets["ticket_id"].duplicated().sum()

np.int64(616)

In [157]:
dup_rows = tickets[tickets["ticket_id"].duplicated(keep=False)]

dup_rows[["ticket_id", "source_system", "csat_score"]].sort_values("ticket_id").head(20)

,ticket_id,source_system,csat_score
12,TK-240016,helpdesk,NaN
13,TK-240016,legacy_fd,0.0
15,TK-240018,helpdesk,NaN
16,TK-240018,legacy_fd,0.0
19,TK-240021,helpdesk,NaN
20,TK-240021,legacy_fd,0.0
23,TK-240025,helpdesk,NaN
24,TK-240025,legacy_fd,0.0
30,TK-240030,legacy_fd,0.0
29,TK-240030,helpdesk,NaN


This asks:

“For each row, does this ticket_id appear more than once?”

keep=False is important: both copies are considered. Then keep only those duplicate rows

In [158]:
len(dup_rows) # 1232 is the output: as both the first copy and its duplicate are considered (616 + 616)

1232

#### Inspect one duplicate closely

In [159]:
sample_id = dup_rows["ticket_id"].iloc[0]

tickets[tickets["ticket_id"] == sample_id].T # .T transposes the table so the fields become rows. Much easier to inspect.

,12,13
ticket_id,TK-240016,TK-240016
created_at,2025-01-03 03:38,2025-01-03 03:38
first_response_at,2025-01-03 03:57,2025-01-03 03:57
resolved_at,2025-01-03 04:25,2025-01-03 04:25
status,resolved,resolved
channel,voice,voice
customer_id,C103315,C103315
order_id,NaN,NaN
product_sku,VA-AC-CH65,VA-AC-CH65
category,Other,Other


##### Closely inspecting the duplicates shows us that they are exactly same in all attributes except 2: 

source_system & csat_score

* csat_score: [NaN, 0.0]
* source_system: [helpdesk, legacy_fd]

The email data and policy explicitly says the current helpdesk went live on 14 Sep 2025 and that legacy tickets were re-imported during reconciliation. Therefore for a case of duplicacy - Given that the helpdesk copy has the normal CSAT representation while the legacy copy can use 0 for no response, the natural canonical choice is:

Keep the helpdesk copy.

#### Code check of whether the duplicates differ in any other volumns or not

In [160]:
compare_cols = [
    c for c in tickets.columns
    if c not in ["source_system", "csat_score"]
]

# We check if the duplicates are identical in all other columns except the 2 listed columns
pair = tickets[tickets["ticket_id"] == sample_id]
same = (
    pair.iloc[0][compare_cols].eq(pair.iloc[1][compare_cols])
    |
    (pair.iloc[0][compare_cols].isna() &
     pair.iloc[1][compare_cols].isna())
)

same.all()

np.True_

In [161]:
dup_rows.groupby("source_system")["ticket_id"].nunique()

source_system
helpdesk     616
legacy_fd    616
Name: ticket_id, dtype: int64

#### Deduplication

Now we need to implement the canonical rule:

For duplicated ticket_ids, retain the helpdesk record

In [162]:
tickets_clean = tickets.copy()

In [163]:
tickets_clean["source_system"].value_counts()

source_system
helpdesk     8079
legacy_fd    3737
Name: count, dtype: int64

In [164]:
# We keep the helpdesk record and remove the legacy_fd one
source_priority = {
    "helpdesk": 0,
    "legacy_fd": 1
}

tickets_clean["_source_priority"] = (
    tickets_clean["source_system"].map(source_priority)
)

In [165]:
tickets_clean = tickets_clean.sort_values(
    ["ticket_id", "_source_priority"]
)

In [166]:
tickets_clean = tickets_clean.drop_duplicates(
    subset="ticket_id",
    keep="first"
)

In [167]:
tickets_clean = tickets_clean.drop(
    columns="_source_priority"
).reset_index(drop=True)

In [168]:
print("Raw rows:", len(tickets))
print("Clean rows:", len(tickets_clean))
print("Unique ticket IDs:", tickets_clean["ticket_id"].nunique())

Raw rows: 11816
Clean rows: 11200
Unique ticket IDs: 11200


In [169]:
if (tickets_clean["ticket_id"].is_unique)==True:
    print("Deduplication successful !!")

Deduplication successful !!


#### Check for Missing Values

But here for missing values we shall neither drop or impute them, since missing vlaues are significantly different and means a different aspect

A missing value for different column might contain:

| Column | Missing means | Action |
|----------|---------------------------|----------------------------------------------|
| `order_id` | Customer didn't provide it | Leave missing |
| `resolved_at` | Ticket unresolved/pending | Leave missing |
| `refund_amount_inr` | No refund | Likely interpret as ₹0 only when doing refund arithmetic |
| `refund_reason_code` | No refund | Leave missing |
| `csat_score` | No response | Exclude from CSAT calculation |


We do not impute missing values globally. We interpret them according to the business meaning of each field.

In [170]:
tickets_clean.isna().sum()

ticket_id                0
created_at               0
first_response_at        0
resolved_at            589
status                   0
channel                  0
customer_id              0
order_id              3840
product_sku              0
category                 0
priority                 0
assigned_team            0
agent_id                 0
transfers                0
csat_score            4477
refund_amount_inr     9222
refund_reason_code    9222
replacement_issued       0
customer_message         0
agent_notes              0
source_system            0
dtype: int64

In [171]:
tickets_clean.isna().mean().mul(100).round(2)

ticket_id              0.00
created_at             0.00
first_response_at      0.00
resolved_at            5.26
status                 0.00
channel                0.00
customer_id            0.00
order_id              34.29
product_sku            0.00
category               0.00
priority               0.00
assigned_team          0.00
agent_id               0.00
transfers              0.00
csat_score            39.97
refund_amount_inr     82.34
refund_reason_code    82.34
replacement_issued     0.00
customer_message       0.00
agent_notes            0.00
source_system          0.00
dtype: float64

#### Next Challenge to Address

This is the next critical issue because:

1. CSV timestamps = UTC
2. shifts = IST
3. SLA is measured from ticket creation to first human response
4. weekly reporting needs the operational local date/week

The policy confirms the API exports UTC while the helpdesk's standard reports/shifts are IST.

In [172]:
tickets_clean[["created_at", "first_response_at"]].dtypes

created_at           str
first_response_at    str
dtype: object

Converting the str object into date-tie format

In [173]:
tickets_clean["created_at"] = pd.to_datetime(
    tickets_clean["created_at"],
    utc=True
)

tickets_clean["first_response_at"] = pd.to_datetime(
    tickets_clean["first_response_at"],
    utc=True
)

**NOTE**

Converting this into pandas datetime object considers them as UTC (utc=True). Later for uniformity we need to convert these into IST.

In [174]:
tickets_clean[["created_at", "first_response_at"]].head()

,created_at,first_response_at
0,2025-01-01 06:12:00+00:00,2025-01-01 06:20:00+00:00
1,2025-01-01 09:01:00+00:00,2025-01-01 09:34:00+00:00
2,2025-01-02 00:41:00+00:00,2025-01-02 03:41:00+00:00
3,2025-01-02 02:21:00+00:00,2025-01-02 03:38:00+00:00
4,2025-01-02 04:05:00+00:00,2025-01-02 04:38:00+00:00


In [175]:
tickets_clean[["created_at", "first_response_at"]].dtypes

created_at           datetime64[us, UTC]
first_response_at    datetime64[us, UTC]
dtype: object

We don't overwrite UTC column rather create a new one

In [176]:
tickets_clean["created_at_ist"] = (
    tickets_clean["created_at"].dt.tz_convert("Asia/Kolkata")
)

tickets_clean["first_response_at_ist"] = (
    tickets_clean["first_response_at"].dt.tz_convert("Asia/Kolkata")
)

In [177]:
tickets_clean[
    ["created_at", "created_at_ist",
     "first_response_at", "first_response_at_ist"]
].head()

,created_at,created_at_ist,first_response_at,first_response_at_ist
0,2025-01-01 06:12:00+00:00,2025-01-01 11:42:00+05:30,2025-01-01 06:20:00+00:00,2025-01-01 11:50:00+05:30
1,2025-01-01 09:01:00+00:00,2025-01-01 14:31:00+05:30,2025-01-01 09:34:00+00:00,2025-01-01 15:04:00+05:30
2,2025-01-02 00:41:00+00:00,2025-01-02 06:11:00+05:30,2025-01-02 03:41:00+00:00,2025-01-02 09:11:00+05:30
3,2025-01-02 02:21:00+00:00,2025-01-02 07:51:00+05:30,2025-01-02 03:38:00+00:00,2025-01-02 09:08:00+05:30
4,2025-01-02 04:05:00+00:00,2025-01-02 09:35:00+05:30,2025-01-02 04:38:00+00:00,2025-01-02 10:08:00+05:30


Calculate Response Duration

In [178]:
tickets_clean["response_time"] = (
    tickets_clean["first_response_at"]
    - tickets_clean["created_at"]
)

In [179]:
tickets_clean["response_minutes"] = (
    tickets_clean["response_time"].dt.total_seconds() / 60
)

In [180]:
tickets_clean["response_minutes"].describe()

count    11200.000000
mean       155.174643
std        229.881160
min          0.000000
25%          7.000000
50%         51.000000
75%        241.000000
max       5155.000000
Name: response_minutes, dtype: float64

Apply the actual SLA policy

In [181]:
sla_minutes = {
    "chat": 15,
    "voice": 120,
    "social": 240,
    "email": 480
}

tickets_clean["sla_target_minutes"] = (
    tickets_clean["channel"].map(sla_minutes)
)

tickets_clean[
    ["channel", "sla_target_minutes"]
].drop_duplicates().sort_values("channel")

,channel,sla_target_minutes
7,chat,15
2,email,480
1,social,240
0,voice,120


Calculate Breach

In [182]:
tickets_clean["breach"] = (
    tickets_clean["response_minutes"]
    > tickets_clean["sla_target_minutes"]
)

tickets_clean["breach"].value_counts()

breach
False    8760
True     2440
Name: count, dtype: int64

In [183]:
tickets_clean["breach"].mean()

np.float64(0.21785714285714286)

#### Validation

For each channel, how many tickets met/missed SLA?

In [184]:
pd.crosstab(
    tickets_clean["channel"],
    tickets_clean["breach"]
)

breach,False,True
channel,,
chat,3591,1368
email,2902,745
social,863,243
voice,1404,84


In [185]:
tickets_clean.groupby("channel")["response_minutes"].agg(
    ["count", "mean", "median", "max"]
)

,count,mean,median,max
channel,,,,
chat,4959,85.151240,6.0,552.0
email,3647,295.752399,216.0,5155.0
social,1106,152.216998,84.0,920.0
voice,1488,46.189516,34.0,413.0


#### Here we add the Roster data - Roster Mapping

In [186]:
agents = pd.read_csv(DATA / "agents.csv")

In [187]:
agents.head()

,agent_id,name,site,team,shift,tier,from_date,to_date
0,A3001,Zoya Mehta,Indore,Chat Frontline,Night,1,2021-04-20,2025-06-29
1,A3002,Tarun Mishra,Indore,Chat Frontline,Night,1,2023-02-12,2025-06-29
2,A3002,Tarun Mishra,Indore,Chat Frontline,Day,1,2025-06-30,NaN
3,A3003,Harpreet Deshpande,Indore,Chat Frontline,Night,1,2023-06-14,2025-06-29
4,A3003,Harpreet Deshpande,Indore,Chat Frontline,Day,1,2025-06-30,NaN


In [188]:
agents.columns

Index(['agent_id', 'name', 'site', 'team', 'shift', 'tier', 'from_date',
       'to_date'],
      dtype='str')

In [189]:
agents[["agent_id", "site", "team", "shift", "tier",
        "from_date", "to_date"]][:5]

,agent_id,site,team,shift,tier,from_date,to_date
0,A3001,Indore,Chat Frontline,Night,1,2021-04-20,2025-06-29
1,A3002,Indore,Chat Frontline,Night,1,2023-02-12,2025-06-29
2,A3002,Indore,Chat Frontline,Day,1,2025-06-30,NaN
3,A3003,Indore,Chat Frontline,Night,1,2023-06-14,2025-06-29
4,A3003,Indore,Chat Frontline,Day,1,2025-06-30,NaN


Now to perform a join of the roster data we need to extract the agent_id from tickets.csv on which we shall join..But agent_id alone isn't enough because an agent can change assignment.

So we need:

agent_id+date→site, shift, team, tier

This is therefore a temporal raster join

#### NOTE

One important decision before the join

Because the policy says: "breaches are reported against the resolving agent."

our roster lookup should use:

agent_id
+
resolved_at

for resolved/closed tickets.

Not:

created_at

because the ticket may have been created before the eventual resolver's assignment.

In [190]:
agents["from_date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [191]:
agents["from_date"] = pd.to_datetime(agents["from_date"])
agents["to_date"] = pd.to_datetime(agents["to_date"])

agents[
    ["agent_id", "site", "team", "shift", "tier",
     "from_date", "to_date"]
].sort_values(["agent_id", "from_date"])

,agent_id,site,team,shift,tier,from_date,to_date
0,A3001,Indore,Chat Frontline,Night,1,2021-04-20,2025-06-29
1,A3002,Indore,Chat Frontline,Night,1,2023-02-12,2025-06-29
2,A3002,Indore,Chat Frontline,Day,1,2025-06-30,NaT
3,A3003,Indore,Chat Frontline,Night,1,2023-06-14,2025-06-29
4,A3003,Indore,Chat Frontline,Day,1,2025-06-30,NaT
5,A3004,Bengaluru,Chat Frontline,Day,1,2022-12-30,NaT
6,A3005,Bengaluru,Chat Frontline,Morning,1,2021-07-28,NaT
7,A3006,Bengaluru,Chat Frontline,Day,1,2021-02-05,NaT
8,A3007,Bengaluru,Chat Frontline,Morning,1,2023-09-10,NaT
9,A3008,Indore,Chat Frontline,Morning,1,2022-05-02,NaT


In [192]:
agents["to_date"].dtype

dtype('<M8[us]')

In [193]:
tickets_clean.columns

Index(['ticket_id', 'created_at', 'first_response_at', 'resolved_at', 'status',
       'channel', 'customer_id', 'order_id', 'product_sku', 'category',
       'priority', 'assigned_team', 'agent_id', 'transfers', 'csat_score',
       'refund_amount_inr', 'refund_reason_code', 'replacement_issued',
       'customer_message', 'agent_notes', 'source_system', 'created_at_ist',
       'first_response_at_ist', 'response_time', 'response_minutes',
       'sla_target_minutes', 'breach'],
      dtype='str')

In [194]:
tickets_clean["resolved_at"]

0        2025-01-02 06:33
1        2025-01-01 09:45
2        2025-01-04 03:59
3        2025-01-12 04:37
4        2025-01-02 05:09
               ...       
11195    2026-06-30 11:25
11196    2026-07-04 04:52
11197                 NaN
11198    2026-06-30 16:08
11199    2026-06-30 16:17
Name: resolved_at, Length: 11200, dtype: str

In [195]:
tickets_clean["resolved_at_utc"] = pd.to_datetime(
    tickets_clean["resolved_at"],
    utc=True
)

In [196]:
tickets_clean["resolved_at_utc"].dtype

datetime64[us, UTC]

In [197]:
tickets_clean["resolved_at_ist"] = (
    tickets_clean["resolved_at_utc"]
    .dt.tz_convert("Asia/Kolkata")
)

In [198]:
tickets_clean[
    ["ticket_id", "agent_id", "resolved_at", "resolved_at_ist", "status"]
].head()

,ticket_id,agent_id,resolved_at,resolved_at_ist,status
0,TK-240001,A3033,2025-01-02 06:33,2025-01-02 12:03:00+05:30,resolved
1,TK-240002,A3015,2025-01-01 09:45,2025-01-01 15:15:00+05:30,resolved
2,TK-240004,A3028,2025-01-04 03:59,2025-01-04 09:29:00+05:30,resolved
3,TK-240006,A3044,2025-01-12 04:37,2025-01-12 10:07:00+05:30,resolved
4,TK-240007,A3035,2025-01-02 05:09,2025-01-02 10:39:00+05:30,resolved


Why are we using resolved_at?

Because the policy explicitly says: breaches are reported against the resolving agent.

And agents.csv tells us which assignment(shift/site/team etc) that resolver was working under at that time.

In [199]:
tickets_clean["status"].value_counts()

status
resolved    9588
closed      1023
open         353
pending      236
Name: count, dtype: int64

In [200]:
tickets_clean["status"].isin(["resolved", "closed"]).sum()

np.int64(10611)

In [201]:
tickets_clean["status"].isin(["open", "pending"]).sum()

np.int64(589)

In [202]:
tickets_clean["resolved_date"] = (
    tickets_clean["resolved_at_ist"].dt.tz_localize(None).dt.normalize()
)

#### Sampele Inspection

Before doing a full join, let's manually solve one ticket.

In [203]:
agent = "A3002"

agents[agents["agent_id"] == agent]

,agent_id,name,site,team,shift,tier,from_date,to_date
1,A3002,Tarun Mishra,Indore,Chat Frontline,Night,1,2023-02-12,2025-06-29
2,A3002,Tarun Mishra,Indore,Chat Frontline,Day,1,2025-06-30,NaT


In [204]:
tickets_clean[
    (tickets_clean["agent_id"] == agent) &
    (tickets_clean["status"].isin(["resolved", "closed"]))
][
    ["ticket_id", "agent_id", "resolved_at_ist"]
].head()

,ticket_id,agent_id,resolved_at_ist
15,TK-240019,A3002,2025-01-04 01:18:00+05:30
17,TK-240021,A3002,2025-01-04 03:43:00+05:30
37,TK-240044,A3002,2025-01-07 02:37:00+05:30
57,TK-240068,A3002,2025-01-14 01:50:00+05:30
61,TK-240072,A3002,2025-01-11 00:23:00+05:30


#### Map each ticket to the resolver's roster info

Isolate completed tickets

In [205]:
completed = tickets_clean[
    tickets_clean["status"].isin(["resolved", "closed"])
].copy()

print(completed.shape)

(10611, 30)


Process roster dates

In [206]:
agents["from_date"] = pd.to_datetime(agents["from_date"]).dt.normalize()
agents["to_date"] = pd.to_datetime(agents["to_date"]).dt.normalize()

In [214]:
completed["resolved_date"] = (
    completed["resolved_at_ist"]
    .dt.tz_localize(None)
    .dt.normalize()
)

#### The Actual Match Rule

For every ticket we want a roster row where:

same agent_id
AND
from_date <= resolved_date
AND
(to_date is missing OR resolved_date <= to_date)

In mathematical form:

$$ from\_date \le resolved\_date \le to\_date $$

with NaT meaning:

to_date=no end date

#### Implementing the Candidate join

In [215]:
candidate = completed.merge(
    agents,
    on="agent_id",
    how="left",
    suffixes=("", "_roster")
)

In [216]:
completed.shape

(10611, 30)

#### Keeping only valid historical roster assignments

In [219]:
valid = candidate[
    (candidate["resolved_date"] >= candidate["from_date"]) &
    (
        candidate["to_date"].isna() |
        (candidate["resolved_date"] <= candidate["to_date"])
    )
].copy()

In [220]:
match_counts = valid["ticket_id"].value_counts()

match_counts.value_counts()

count
1    10611
Name: count, dtype: int64

In [221]:
(match_counts == 1).all()

np.True_

#### Next: We address the Business Questions

#####  Channel Analysis

We try studying

1. Overall SLA
2. Channel SLA
3. Shift SLA
4. Shift × Channel
5. Weekly trend
6. Agent analysis
7. Root cause
8. Cost

In [222]:
# we use the tickets_clean and not valid df, since every unique ticket has a first response, so every 11,200 canonical tickets can participate in the SLA calculation.

channel_summary = (
    tickets_clean
    .groupby("channel")
    .agg(
        tickets=("ticket_id", "count"),
        breaches=("breach", "sum"),
        breach_rate=("breach", "mean")
    )
)

channel_summary["breach_rate"] *= 100

channel_summary

,tickets,breaches,breach_rate
channel,,,
chat,4959,1368,27.586207
email,3647,745,20.427749
social,1106,243,21.971067
voice,1488,84,5.645161


In [223]:
channel_summary.sort_values("breach_rate", ascending=False)

,tickets,breaches,breach_rate
channel,,,
chat,4959,1368,27.586207
social,1106,243,21.971067
email,3647,745,20.427749
voice,1488,84,5.645161


In [225]:
tickets_clean["sla_target_minutes"].isna().sum()

np.int64(0)

**Shift SLA**

Among completed tickets attributed to the resolving agent, how does first-response SLA performance differ by resolver shift?

In [226]:
# Here we must use valid df since shift comes from the resolver's roster.

shift_summary = (
    valid
    .groupby("shift")
    .agg(
        tickets=("ticket_id", "count"),
        breaches=("breach", "sum"),
        breach_rate=("breach", "mean")
    )
)

shift_summary["breach_rate"] *= 100

shift_summary

,tickets,breaches,breach_rate
shift,,,
Day,4428,376,8.491418
Morning,5928,1917,32.338057
Night,255,27,10.588235


**Shift x Channel**

For each shift, what percentage of its tickets come from each channel?

In [227]:
shift_channel = (
    valid
    .groupby(["shift", "channel"])
    .agg(
        tickets=("ticket_id", "count"),
        breaches=("breach", "sum"),
        breach_rate=("breach", "mean")
    )
)

shift_channel["breach_rate"] *= 100

shift_channel

tickets  breaches  breach_rate
shift   channel                                
Day     chat        1918       183     9.541189
        email       1258       112     8.903021
        social       424        36     8.490566
        voice        828        45     5.434783
Morning chat        2699      1115    41.311597
        email       2092       583    27.868069
        social       577       185    32.062392
        voice        560        34     6.071429
Night   chat         104         7     6.730769
        email        112        14    12.500000
        social        31         4    12.903226
        voice          8         2    25.000000

In [228]:
channel_mix = pd.crosstab(
    valid["shift"],
    valid["channel"],
    normalize="index"
) * 100

channel_mix.round(2)

channel,chat,email,social,voice
shift,,,,
Day,43.32,28.41,9.58,18.70
Morning,45.53,35.29,9.73,9.45
Night,40.78,43.92,12.16,3.14
